In [ ]:
# Colab bootstrap — auto-clone repo on Google Colab, no-op locally
import os, sys, subprocess

REPO = "https://github.com/jongmoonha/AI-PHM_Graduate.git"
DIR  = "AI-PHM_Graduate"

try:
    import google.colab  # type: ignore
    target = '/content/' + DIR
    if not os.path.isdir(target):
        subprocess.run(["git", "clone", REPO, target], check=True)
    os.chdir(target)
    print('Google Colab detected. Working directory:', os.getcwd())
except ImportError:
    print('Local environment detected. Working directory:', os.getcwd())


# 푸리에 급수 — Python 실습

## 학습 목표
- 복소 푸리에 계수 $c_n$을 해석적 적분(SymPy)으로 직접 구한다
- 고조파 번호 $n$과 실제 주파수 $f = nf_0$의 관계를 이해한다
- 단순한 신호에서 복잡한 신호로 갈수록 더 많은 $c_n$이 필요함을 확인한다
- 수식으로 만든 사각파를 실제 FFT로 검증하여 이론과 실측을 연결한다

In [ ]:
# Colab 환경 설정 (로컬에서는 아무 작업 없음)
import os, sys

if 'google.colab' in sys.modules:
    REPO = 'AI-PHM_Graduate'
    if not os.path.exists(REPO):
        !git clone -q https://github.com/jongmoonha/AI-PHM_Graduate.git
    sys.path.insert(0, f'/content/{REPO}')
    # 한글 폰트 설치 (matplotlib 글리프 경고 방지)
    !apt-get install -y -qq fonts-nanum > /dev/null
    import matplotlib.font_manager as fm
    for fp in fm.findSystemFonts(fontpaths=['/usr/share/fonts/truetype/nanum']):
        fm.fontManager.addfont(fp)

> (**로컬 실행 시 주의**: `Signal_Processing_Practice_from_Basics_to_Advanced/` 디렉터리를 작업 폴더로 Jupyter를 실행해야 `utils.py`를 자동으로 import할 수 있다.)

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

---

## 공통 도구: SymPy로 $c_n$ 계산

복소 푸리에 계수의 정의:

$$c_n = \frac{1}{T}\int_0^T x(t)\, e^{-jn\omega_0 t}\, dt \qquad (\omega_0 = 2\pi f_0,\; T = 1/f_0)$$

아래 함수는 이 적분을 SymPy로 **해석적**으로 수행한다.

In [ ]:
t = sp.Symbol('t', real=True)

def compute_cn(x, f0=1, n_range=5):
    """c_n을 -n_range ~ +n_range까지 해석적으로 계산한다."""
    f0_sym = sp.Integer(f0)
    T = 1 / f0_sym
    w0 = 2 * sp.pi * f0_sym
    cn = {}
    for k in range(-n_range, n_range + 1):
        c_k = (1/T) * sp.integrate(x * sp.exp(-sp.I * k * w0 * t), (t, 0, T))
        cn[k] = c_k
    return cn

In [ ]:
def plot_cn(cn, f0, title=""):
    """c_n을 Hz축 스템 플롯으로 표시 (진폭과 위상)."""
    ns = np.array(sorted(cn.keys()))
    mags = np.array([float(abs(cn[n])) for n in ns])
    phases = np.array([float(sp.arg(cn[n])) if abs(cn[n]) > 1e-12 else 0.0 for n in ns])
    freqs = ns * f0

    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    ax[0].stem(freqs, mags, basefmt=' ')
    ax[0].set_xlabel('Frequency (Hz)'); ax[0].set_ylabel('$|c_n|$'); ax[0].set_title('Amplitude')
    ax[1].stem(freqs, phases, basefmt=' ')
    ax[1].set_xlabel('Frequency (Hz)'); ax[1].set_ylabel(r'$\angle c_n$ (rad)'); ax[1].set_title('Phase')
    if title:
        fig.suptitle(title, fontsize=12)
    fig.tight_layout(); plt.show()

---

## 실습 1. 스펙트럼 표시 방법 — $n$ 축 vs 주파수(Hz) 축

$c_n$을 구하면 스펙트럼을 그릴 수 있다. x축을 무엇으로 하느냐에 따라 두 가지 표현이 가능하다.

- **$n$ 축**: 고조파 번호. $c_n$ 값은 $f_0$와 무관하다.
- **주파수(Hz) 축**: $f = n \times f_0$. $f_0$에 따라 피크 위치가 달라진다.

아래에서 $\cos(\omega_0 t)$를 $f_0 = 1$ Hz와 $f_0 = 2$ Hz로 각각 분석하여 비교한다.

In [ ]:
f0 = 1
w0 = 2 * sp.pi * f0
x = sp.cos(w0 * t)
cn = compute_cn(x, f0)

ns = np.array(sorted(cn.keys()))
mags = np.array([float(abs(cn[n])) for n in ns])

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].stem(ns, mags, basefmt=' ')
ax[0].set_xlabel('$n$'); ax[0].set_ylabel('$|c_n|$'); ax[0].set_title('x-axis: n')
ax[1].stem(ns * f0, mags, linefmt='C1-', markerfmt='C1o', basefmt=' ')
ax[1].set_xlabel('Frequency (Hz)'); ax[1].set_ylabel('$|c_n|$'); ax[1].set_title(f'x-axis: f (f0={f0} Hz)')
fig.suptitle(r'$\cos(\omega_0 t),\ f_0 = 1$ Hz')
fig.tight_layout(); plt.show()

In [ ]:
f0 = 2
w0 = 2 * sp.pi * f0
x = sp.cos(w0 * t)
cn = compute_cn(x, f0)

ns = np.array(sorted(cn.keys()))
mags = np.array([float(abs(cn[n])) for n in ns])

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].stem(ns, mags, basefmt=' ')
ax[0].set_xlabel('$n$'); ax[0].set_ylabel('$|c_n|$'); ax[0].set_title('x-axis: n')
ax[1].stem(ns * f0, mags, linefmt='C1-', markerfmt='C1o', basefmt=' ')
ax[1].set_xlabel('Frequency (Hz)'); ax[1].set_ylabel('$|c_n|$'); ax[1].set_title(f'x-axis: f (f0={f0} Hz)')
fig.suptitle(r'$\cos(\omega_0 t),\ f_0 = 2$ Hz')
fig.tight_layout(); plt.show()

- **$n$ 축 (왼쪽)**: $f_0$가 달라도 $c_{\pm1} = 1/2$로 동일. $c_n$은 파형의 **형태**만 반영한다.
- **Hz 축 (오른쪽)**: $f_0 = 1$이면 $\pm1$ Hz, $f_0 = 2$이면 $\pm2$ Hz에 피크. 간격이 $f_0$만큼 벌어진다.

이후 실습에서는 **주파수(Hz) 축**으로 표시한다.

---

## 실습 2. $e^{+j\omega_0 t}$ — 양의 주파수 기저 함수

복소지수함수는 푸리에 급수의 **기저 함수** 그 자체다.
$e^{j\omega_0 t}$는 이미 "1개의 항"이므로, $c_1 = 1$이고 나머지는 모두 0이어야 한다.

In [ ]:
f0 = 1
w0 = 2 * sp.pi * f0

x = sp.exp(sp.I * w0 * t)
cn = compute_cn(x, f0)

for n, c in cn.items():
    print(f"n={n:+d}: {c}")

In [ ]:
plot_cn(cn, f0, title=r'$e^{+j\omega_0 t}$')

$c_{+1} = 1$만 존재한다. $+f_0$에 피크 1개.

### 직접 해보기

1. $e^{j \cdot 3\omega_0 t}$의 계수를 구하면 $c_3 = 1$이 나오는가?
2. $2 e^{j\omega_0 t}$이면 $c_1$이 어떻게 달라지는가?

---

## 실습 3. $e^{-j\omega_0 t}$ — 음의 주파수

부호가 반대이면 음의 주파수에 피크가 나타나야 한다.

In [ ]:
f0 = 1
w0 = 2 * sp.pi * f0

x = sp.exp(-sp.I * w0 * t)
cn = compute_cn(x, f0)

for n, c in cn.items():
    print(f'n={n:+d}: {c}')

In [ ]:
plot_cn(cn, f0, title=r'$e^{-j\omega_0 t}$')

$c_{-1} = 1$만 존재한다. $-f_0$에 피크 1개.

$e^{+j\omega_0 t}$와 $e^{-j\omega_0 t}$는 푸리에 급수의 "원자"이다. 다른 모든 주기 신호는 이들의 조합이다.

### 직접 해보기

1. $e^{-j \cdot 2\omega_0 t}$이면 피크가 어디에 나타나는가?
2. $e^{+j\omega_0 t} + e^{-j\omega_0 t}$의 계수를 구해보자. 결과가 익숙한가?

---

## 실습 4. $\cos(\omega_0 t)$ — 실수 신호의 양면 스펙트럼

오일러 공식에 의해:

$$\cos(\omega_0 t) = \frac{1}{2}e^{+j\omega_0 t} + \frac{1}{2}e^{-j\omega_0 t}$$

실습 1, 2의 결과를 반씩 합치면 $c_{+1} = c_{-1} = 1/2$가 되어야 한다.

In [ ]:
f0 = 1
w0 = 2 * sp.pi * f0

x = sp.cos(w0 * t)
cn = compute_cn(x, f0)

for n, c in cn.items():
    print(f'n={n:+d}: {c}')

In [ ]:
plot_cn(cn, f0, title=r'$\cos(\omega_0 t)$')

- $c_{\pm1} = 1/2$: 진폭 1의 **정확히 절반**씩 양쪽에 나뉜다.
- 이것이 실수 신호의 스펙트럼이 $\pm f_0$에 피크를 갖는 이유다.
- 실수 신호 → 항상 $c_{-n} = c_n^*$ (켤레 대칭).

---

## 실습 5. $A\cos(\omega_0 t + \phi)$ — 진폭·위상이 있는 경우

$x(t) = A\cos(\omega_0 t + \phi)$이면 $c_{\pm1} = \frac{A}{2}e^{\pm j\phi}$이다.

In [ ]:
f0 = 100
w0 = 2 * sp.pi * f0

x = 3 * sp.cos(w0 * t + sp.pi / 6)
cn = compute_cn(x, f0)

for n, c in cn.items():
    print(f'n={n:+d}: {c}')

In [ ]:
plot_cn(cn, f0, title=r'$3\cos(\omega_0 t + \pi/6)$')

- $|c_{\pm1}| = 3/2$: 원래 진폭 3의 절반
- $\angle c_1 = +\pi/6$, $\angle c_{-1} = -\pi/6$: 초기 위상이 그대로 반영
- 켤레 대칭 $c_{-1} = c_1^*$ 확인

### 직접 해보기

1. $\sin(\omega_0 t)$의 계수를 구해보자. $\cos$과 비교했을 때 위상이 어떻게 다른가?
2. $\cos(\omega_0 t) + \cos(3\omega_0 t)$이면 0이 아닌 $c_n$은 몇 개인가?

---

## 실습 6. 사각파 — 무한히 많은 고조파가 필요한 신호

진폭 $\pm1$, 주기 $T$인 사각파:

$$x(t) = \begin{cases} +1 & 0 < t < T/2 \\ -1 & T/2 < t < T \end{cases}$$

이 신호는 홀수 고조파만 포함하며, 계수는 $1/n$으로 감소한다.

> (SymPy 해석 적분이므로 `n_range`를 크게 잡으면 수 초 소요된다.)

In [ ]:
f0 = 1
w0 = 2 * sp.pi * f0
T = sp.Integer(1) / sp.Integer(f0)

x = sp.Piecewise((1, t < T / 2), (-1, True))
cn = compute_cn(x, f0, n_range=10)

for n, c in cn.items():
    print(f'n={n:+d}: {c}')

In [ ]:
plot_cn(cn, f0, title='Square Wave')

- 홀수 $n$에서만 $c_n \neq 0$: 사각파의 반파 대칭 때문이다.
- $|c_n| = 2/(\pi n)$으로 감소 — 불연속점이 있으면 $1/n$ 감소.
- 위상은 $\pm\pi/2$: 순수 사인 성분 ($c_n$이 순허수).

### 부분합: 항을 더할수록 사각파에 수렴

In [ ]:
def square_partial_sum(t_arr, f0, N):
    """사각파의 푸리에 부분합 (홀수 고조파만 포함)."""
    y = np.zeros_like(t_arr)
    for k in range(1, N + 1, 2):
        y += (4 / (np.pi * k)) * np.sin(2 * np.pi * k * f0 * t_arr)
    return y

f0 = 1
t_num = np.linspace(-0.1, 2.1, 4000)
square = np.sign(np.sin(2 * np.pi * f0 * t_num))

fig, ax = plt.subplots(2, 2, figsize=(12, 6))
for axi, N in zip(ax.flat, [1, 5, 21, 99]):
    axi.plot(t_num, square, 'k--', alpha=0.4, label='Square Wave')
    axi.plot(t_num, square_partial_sum(t_num, f0, N), 'C0', label=f'N={N}')
    axi.set_xlim(0, 2); axi.set_ylim(-1.4, 1.4)
    axi.set_xlabel('t (s)'); axi.set_ylabel('x(t)')
    axi.set_title(f'N={N}')
    axi.legend(loc='upper right')
fig.tight_layout(); plt.show()

- $N=1$: 순수 사인파 1개
- $N=99$: 거의 사각파이지만, 불연속점에서 오버슈트가 남아 있다

이 오버슈트가 **Gibbs 현상**이다. 항 수를 아무리 늘려도 불연속 점프($=2.0$)의 약 **9%** 가 오버슈트로 남는다 — $+1$ 쪽에서 약 $1.09$ 까지 튄다.

> 양측 스펙트럼의 $|c_n| = 2/(\pi n)$은 단측 진폭 스펙트럼에서 $4/(\pi k)$로 두 배가 되어 보인다.

### 직접 해보기

1. 삼각파(`sp.Piecewise((4*t/T - 1, t < T/2), (3 - 4*t/T, True))`)의 $c_n$을 구해보자. $1/n^2$으로 감소하는가?
2. N=999를 넣으면 Gibbs 오버슈트가 줄어드는가?
3. `f0`를 2로 바꾸고 다시 실행해보자. Hz축 스펙트럼이 어떻게 달라지는가?

---

## 실습 7. 부분합의 FFT — 이론과 실측 연결

지금까지는 SymPy로 이론적 $c_n$을 구했다. 이번엔 반대로 해본다.

1. 푸리에 급수 부분합(N=5, 10, 50)으로 사각파 근사 신호를 **시간 도메인**에서 만든다
2. `utils.fft(x, fs)`로 **주파수 도메인** 스펙트럼을 얻는다
3. 이론값($|c_n| = 2/(\pi n)$의 2배인 $4/(\pi n)$, 단측이므로 $\sin$ 기반 진폭) 피크와 일치하는지 눈으로 확인한다

이 한 단계가 **수식 세계(SymPy)와 신호 세계(NumPy + FFT)**를 이어준다.

> (`utils.fft`는 **단측 진폭 스펙트럼**을 반환한다.)

**메모**: 실수 신호이므로 $c_{-n}$ 과 $c_{+n}$ 이 단측 $|X(f)|$ 에서 하나의 피크로 합쳐져 $|c_n|$ 의 **2배**인 $\dfrac{4}{\pi n}$ 로 나타난다.

In [ ]:
from utils import fft

fs = 1000
T_total = 1.0
t_arr = np.arange(0, T_total, 1/fs)
f0 = 5

fig, ax = plt.subplots(3, 2, figsize=(12, 8))
for row, N in enumerate([5, 10, 50]):
    x = np.zeros_like(t_arr)
    for k in range(1, N + 1, 2):
        x += (4 / (np.pi * k)) * np.sin(2 * np.pi * k * f0 * t_arr)
    f_spec, A_spec = fft(x, fs)

    ax[row, 0].plot(t_arr, x, 'C0')
    ax[row, 0].set_xlim(0, 0.4); ax[row, 0].set_ylim(-1.4, 1.4)
    ax[row, 0].set_xlabel('Time (s)'); ax[row, 0].set_ylabel('Amplitude')
    ax[row, 0].set_title(f'Square Wave Partial Sum (N={N})')

    ax[row, 1].stem(f_spec, A_spec, basefmt=' ')
    ax[row, 1].set_xlim(0, 100); ax[row, 1].set_xlabel('Frequency (Hz)'); ax[row, 1].set_ylabel('|X(f)|')
    ax[row, 1].set_title(f'FFT Spectrum (N={N})')
fig.tight_layout(); plt.show()

- 왼쪽 시간 파형: $N$이 커질수록 사각파에 가까워진다.
- 오른쪽 FFT: $f_0 = 5, 3f_0 = 15, 5f_0 = 25, \dots$의 **홀수 고조파**에만 피크가 나타난다.
- 피크 높이는 $4/(\pi k)$에 가까워야 한다 (단측 진폭 스펙트럼). $k=1$이면 약 1.27, $k=3$이면 약 0.42.
- $N$을 늘리면 더 많은 고조파 피크가 추가될 뿐, 기존 피크 높이는 거의 그대로다.

> 고조파는 `N=50`에서 최대 245 Hz까지 존재하지만, 본 그림은 **100 Hz 이상을 생략**하여 저차 고조파의 감쇠 추세에 집중한다.

### 이론과 실측의 다리

SymPy가 알려준 계수($c_n = -2j/(\pi n)$)를 **NumPy로 합산**해 신호를 만들고, **FFT로 분해**하면 똑같은 계수가 되돌아온다. 이것이 푸리에 분석의 완결성이다.

---

## 정리

| 신호 | 0이 아닌 $c_n$ | 핵심 |
|------|---------------|------|
| $e^{j\omega_0 t}$ | $c_1 = 1$ | 기저 함수 자체 |
| $e^{-j\omega_0 t}$ | $c_{-1} = 1$ | 음의 주파수 기저 |
| $\cos(\omega_0 t)$ | $c_{\pm1} = 1/2$ | 실수 = 양쪽 회전의 합 |
| 사각파 | $c_n = -\dfrac{2j}{\pi n}$ ($n>0$, 홀수), $c_{-n}=c_n^*$ | 불연속 → $1/n$ 감소 |

$c_n$은 파형의 **형태**를 나타내고, 실제 주파수는 $f = n \times f_0$으로 결정된다.

### 핵심 수식

$$c_n = \frac{1}{T}\int_0^T x(t)\, e^{-jn\omega_0 t}\, dt$$
